Nama Kelompok:

Ihsan Maulana Preasetia (24523241)

Maulana Shiddiq Afdhaluddin (24523022)

## Preprocessing Data yang Diperlukan

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# Load the dataset
df = pd.read_csv('/content/invistico_Airline.csv')

# Impute missing values for 'Arrival Delay in Minutes' with the mean
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].mean())
print("Missing values after imputation:")
display(df.isnull().sum())

Missing values after imputation:


/tmp/ipykernel_3807/3718659182.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].mean(), inplace=True)


,0
satisfaction,0
Customer Type,0
Age,0
Type of Travel,0
Class,0
Flight Distance,0
Seat comfort,0
Departure/Arrival time convenient,0
Food and drink,0
Gate location,0


Sekarang, kita akan mengkodekan fitur kategorikal menjadi representasi numerik. Untuk fitur independen, kita akan menggunakan One-Hot Encoding, dan untuk variabel target `satisfaction`, kita akan menggunakan Label Encoding.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Label encode the target variable 'satisfaction'
le = LabelEncoder()
df['satisfaction_encoded'] = le.fit_transform(df['satisfaction'])

# Columns to drop: 'satisfaction' is always dropped
columns_to_drop = ['satisfaction']

# Conditionally add 'id' to columns_to_drop if it exists in the DataFrame
if 'id' in df.columns:
    columns_to_drop.append('id')

# Drop the specified columns
df_processed = df.drop(columns=columns_to_drop)

# Identify categorical and numerical columns for one-hot encoding
categorical_cols = df_processed.select_dtypes(include=['object']).columns

# One-hot encode the remaining categorical features
df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

print("Processed DataFrame head:")
display(df_processed.head())
print("\nProcessed DataFrame info:")
df_processed.info()

## 3. Pembagian Data Training dan Testing

In [ ]:
# Separate features (X) and target (y)
X = df_processed.drop('satisfaction_encoded', axis=1)
y = df_processed['satisfaction_encoded']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("\nDistribution of target in training set:")
display(y_train.value_counts(normalize=True))
print("\nDistribution of target in testing set:")
display(y_test.value_counts(normalize=True))

## 3. Membangun Model Logistic Regression

In [ ]:
# Initialize and train the Logistic Regression model
log_reg_model = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)
log_reg_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

## 4. Tampilkan hasil evaluasi model

In [ ]:
# Make predictions on the test set
y_pred = log_reg_model.predict(X_test)

# 4.1 Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# 4.2 Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

## 6. Analisis Kinerja Model dan Kesimpulan

In [17]:
# 1. Kinerja Model Berdasarkan Classification Report:
# Model menunjukkan kinerja yang sangat baik dalam mengklasifikasikan kepuasan pelanggan.
# Precision untuk 'dissatisfied' adalah 0.81, artinya dari semua prediksi 'dissatisfied',
# 81% di antaranya benar. Recall untuk 'dissatisfied' adalah 0.81, yang berarti model dapat
# mengidentifikasi 81% dari semua pelanggan yang sebenarnya 'dissatisfied'.
# Precision untuk 'satisfied' adalah 0.84, dan Recall-nya adalah 0.84, menunjukkan kinerja serupa.
# F1-score (harmonic mean dari precision dan recall) untuk kedua kelas juga tinggi (0.81 dan 0.84),
# menandakan keseimbangan yang baik antara precision dan recall.
# Akurasi keseluruhan model adalah 0.83, menunjukkan bahwa 83% dari semua prediksi model adalah benar.

In [18]:
# 2. Makna True Positive, False Positive, True Negative, dan False Negative:
# Menggunakan label '0' untuk 'dissatisfied' dan '1' untuk 'satisfied':

# True Positive (TP): Model memprediksi 'satisfied' (1) dan pelanggan memang 'satisfied' (1).
# Jumlahnya: 11987 (berada di baris 'satisfied', kolom 'satisfied' pada confusion matrix).
# Artinya, model dengan benar mengidentifikasi 11987 pelanggan yang puas.

# False Positive (FP): Model memprediksi 'satisfied' (1) tetapi pelanggan sebenarnya 'dissatisfied' (0).
# Jumlahnya: 2216 (berada di baris 'dissatisfied', kolom 'satisfied' pada confusion matrix).
# Artinya, model salah mengklasifikasikan 2216 pelanggan yang tidak puas sebagai puas.

# True Negative (TN): Model memprediksi 'dissatisfied' (0) dan pelanggan memang 'dissatisfied' (0).
# Jumlahnya: 9543 (berada di baris 'dissatisfied', kolom 'dissatisfied' pada confusion matrix).
# Artinya, model dengan benar mengidentifikasi 9543 pelanggan yang tidak puas.

# False Negative (FN): Model memprediksi 'dissatisfied' (0) tetapi pelanggan sebenarnya 'satisfied' (1).
# Jumlahnya: 2230 (berada di baris 'satisfied', kolom 'dissatisfied' pada confusion matrix).
# Artinya, model salah mengklasifikasikan 2230 pelanggan yang puas sebagai tidak puas.

# 3. Kesimpulan Mengenai Seberapa Baik Logistic Regression Bekerja:
# Model Logistic Regression menunjukkan kinerja yang sangat baik pada dataset ini.
# Dengan akurasi 83% dan F1-score yang tinggi untuk kedua kelas, model ini mampu
# mengklasifikasikan pelanggan yang puas dan tidak puas dengan tingkat keberhasilan yang tinggi.
# Jumlah False Positives dan False Negatives relatif rendah dibandingkan total sampel,
# menunjukkan bahwa model memiliki kemampuan diskriminasi yang kuat. Secara keseluruhan,
# Logistic Regression adalah pilihan yang efektif untuk tugas klasifikasi ini.

## 5. Tampilkan beberapa contoh hasil prediksi (y_pred) dan bandingkan dengan label sebenarnya (y_test).

In [19]:
# Create a DataFrame to compare actual and predicted values
results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

# Map encoded labels back to original text labels for better readability
results['Actual_Label'] = le.inverse_transform(results['Actual'])
results['Predicted_Label'] = le.inverse_transform(results['Predicted'])

# Display the first 20 examples
print("Comparison of Actual vs. Predicted Labels (First 20 examples):")
display(results.head(20))

Comparison of Actual vs. Predicted Labels (First 20 examples):


,Actual,Predicted,Actual_Label,Predicted_Label
63785,1,1,satisfied,satisfied
86893,1,1,satisfied,satisfied
1072,0,0,dissatisfied,dissatisfied
113145,1,1,satisfied,satisfied
34785,0,0,dissatisfied,dissatisfied
81389,0,0,dissatisfied,dissatisfied
94869,1,1,satisfied,satisfied
83763,0,0,dissatisfied,dissatisfied
3275,1,0,satisfied,dissatisfied
4750,1,0,satisfied,dissatisfied
